In [4]:
import os
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
 
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
from matplotlib.gridspec import GridSpec
import seaborn as sns

 
class SyntheticCohortGenerator:
    CHANNELS = {
        'organic': {'cac': 20, 'arpu': 15, 'weibull_shape': 1.3, 'weibull_scale': 25, 'monthly_signups': 800},
        'referral': {'cac': 25, 'arpu': 14, 'weibull_shape': 1.25, 'weibull_scale': 23, 'monthly_signups': 400},
        'paid_search': {'cac': 45, 'arpu': 16, 'weibull_shape': 0.95, 'weibull_scale': 15, 'monthly_signups': 600},
        'paid_social': {'cac': 70, 'arpu': 15, 'weibull_shape': 0.8, 'weibull_scale': 12, 'monthly_signups': 300},
        'direct': {'cac': 35, 'arpu': 17, 'weibull_shape': 1.15, 'weibull_scale': 20, 'monthly_signups': 500}
    }
 
    def __init__(self, n_months=24, start_date='2023-01-01', discount_rate=0.10):
        self.n_months = n_months
        self.start_date = pd.to_datetime(start_date)
        self.discount_rate = discount_rate
        self.df = None
 
    def weibull_retention(self, months_since_signup, shape, scale):
        return np.exp(-(months_since_signup / scale) ** shape)
 
    def generate(self):
        rows = []
        for cohort_idx in range(self.n_months):
            cohort_date = self.start_date + pd.Timedelta(days=30 * cohort_idx)
            cohort_label = cohort_date.strftime('%Y-%m')
            for channel_name, params in self.CHANNELS.items():
                cohort_size = params['monthly_signups']
                cac = params['cac']
                arpu = params['arpu']
                shape = params['weibull_shape']
                scale = params['weibull_scale']
                for months_since in range(self.n_months - cohort_idx):
                    retention_rate = self.weibull_retention(months_since, shape, scale)
                    active_users = int(cohort_size * retention_rate)
                    monthly_revenue = active_users * arpu
                    rows.append({
                        'cohort_month': cohort_label, 'cohort_date': cohort_date, 'channel': channel_name,
                        'cac': float(cac), 'arpu': float(arpu), 'weibull_shape': shape, 'weibull_scale': scale,
                        'months_since_signup': months_since, 'cohort_size': cohort_size,
                        'active_users': active_users, 'retention_rate': retention_rate,
                        'monthly_revenue': float(monthly_revenue)
                    })
        self.df = pd.DataFrame(rows)
        return self.df
 
    def save_to_csv(self, filename='synthetic_cohorts.csv'):
        if self.df is None:
            self.generate()
        os.makedirs(os.path.dirname(filename) or '.', exist_ok=True)
        self.df.to_csv(filename, index=False)
        return filename
 
 

 
class CohortAnalyzer:
    def __init__(self, cohort_df, discount_rate=0.10):
        self.df = cohort_df.copy()
        self.discount_rate = discount_rate
        self.monthly_discount_rate = (1 + discount_rate) ** (1 / 12) - 1
        self._calculate_retention_rates()
        self._calculate_discounted_revenue()
 
    def _calculate_retention_rates(self):
        self.df['retention_rate'] = (self.df['active_users'] / self.df['cohort_size']).fillna(0)
 
    def _calculate_discounted_revenue(self):
        self.df['discounted_revenue'] = (
            self.df['monthly_revenue'] / ((1 + self.monthly_discount_rate) ** self.df['months_since_signup'])
        )
 
    def retention_matrix(self, channel=None, metric='retention_rate'):
        if channel:
            df = self.df[self.df['channel'] == channel].copy()
        else:
            df = self.df.groupby(['cohort_month', 'months_since_signup']).agg({
                'active_users': 'sum', 'cohort_size': 'sum', 'retention_rate': 'mean'
            }).reset_index()
 
        value_col = 'retention_rate' if metric == 'retention_rate' else 'active_users'
        pivot = df.pivot_table(index='cohort_month', columns='months_since_signup', values=value_col)
        return pivot
 
    def calculate_ltv_by_cohort_channel(self):
        ltv_results = []
        for (cohort, channel), group in self.df.groupby(['cohort_month', 'channel']):
            group = group.sort_values('months_since_signup').reset_index(drop=True)
            cac = group['cac'].iloc[0]
            arpu = group['arpu'].iloc[0]
            ltv = group['discounted_revenue'].sum()
 
            cumulative = 0
            payback_month = None
            for idx, row in group.iterrows():
                cumulative += row['monthly_revenue']
                if cumulative >= cac and payback_month is None:
                    payback_month = row['months_since_signup']
 
            cum_12m = group[group['months_since_signup'] <= 11]['monthly_revenue'].sum()
            ltv_ratio = round(ltv / cac, 2) if cac > 0 else 0
 
            ltv_results.append({
                'cohort_month': cohort, 'channel': channel, 'cac': cac, 'arpu': arpu,
                'ltv': round(ltv, 2), 'ltv_ratio': ltv_ratio, 'payback_month': payback_month,
                'cumulative_revenue_12m': round(cum_12m, 2)
            })
        return pd.DataFrame(ltv_results)
 
    def channel_summary(self):
        ltv_cohorts = self.calculate_ltv_by_cohort_channel()
        summary = ltv_cohorts.groupby('channel').agg({
            'cac': 'first', 'ltv': ['mean', 'std'], 'ltv_ratio': ['mean', 'std'],
            'payback_month': ['mean', 'min', 'max'], 'cohort_month': 'count'
        }).round(2)
        summary.columns = ['_'.join(col).strip('_') for col in summary.columns]
        summary = summary.rename(columns={'cohort_month_count': 'cohort_count'})
        return summary.reset_index()
 
    def payback_analysis(self):
        ltv_cohorts = self.calculate_ltv_by_cohort_channel()
        payback = ltv_cohorts.dropna(subset=['payback_month']).groupby('channel').agg({
            'payback_month': ['min', 'mean', 'max', 'std']
        }).round(1)
        payback.columns = ['_'.join(col).strip('_') for col in payback.columns]
        return payback.reset_index()
 
    def monthly_retention_curve_by_channel(self, channel):
        channel_data = self.df[self.df['channel'] == channel]
        curve = channel_data.groupby('months_since_signup')['retention_rate'].mean()
        return curve.sort_index()
 
    def all_retention_curves(self):
        curves = {}
        for channel in self.df['channel'].unique():
            curves[channel] = self.monthly_retention_curve_by_channel(channel)
        return pd.DataFrame(curves)
 
    def profitability_matrix(self):
        ltv_cohorts = self.calculate_ltv_by_cohort_channel()
        return ltv_cohorts.pivot(index='cohort_month', columns='channel', values='ltv_ratio')
 
 

 
class CohortVisualizer:
    def __init__(self, analyzer, figsize_default=(9, 5.5)):
        self.analyzer = analyzer
        self.figsize_default = figsize_default
        sns.set_style("whitegrid")
 
    def retention_heatmap(self, channel=None):
        retention = self.analyzer.retention_matrix(channel=channel)
        fig, ax = plt.subplots(figsize=self.figsize_default)
        sns.heatmap(retention * 100, cmap='RdYlGn', annot=False, fmt='.0f',
                    cbar_kws={'label': 'Retention Rate (%)'}, ax=ax,
                    linewidths=0.5, vmin=0, vmax=100)
        ax.set_title(f'Cohort Retention Heatmap - {channel.title() if channel else "All Channels"}',
                     fontsize=13, fontweight='bold')
        ax.set_xlabel('Months Since Signup')
        ax.set_ylabel('Cohort (Signup Month)')
        fig.tight_layout()
        return fig
 
    def retention_curves_by_channel(self):
        fig, ax = plt.subplots(figsize=self.figsize_default)
        curves = self.analyzer.all_retention_curves()
        for channel in curves.columns:
            ax.plot(curves.index, curves[channel] * 100, marker='o', linewidth=2,
                    markersize=4, label=channel.title())
        ax.set_title('Retention Curves by Acquisition Channel', fontsize=13, fontweight='bold')
        ax.set_xlabel('Months Since Signup')
        ax.set_ylabel('Retention Rate (%)')
        ax.set_ylim(0, 105)
        ax.grid(True, alpha=0.3)
        ax.legend(loc='upper right', fontsize=9)
        fig.tight_layout()
        return fig
 
    def ltv_by_channel_boxplot(self):
        ltv_data = self.analyzer.calculate_ltv_by_cohort_channel()
        fig, ax = plt.subplots(figsize=self.figsize_default)
        channels = sorted(ltv_data['channel'].unique())
        ltv_by_channel = [ltv_data[ltv_data['channel'] == ch]['ltv'].values for ch in channels]
        bp = ax.boxplot(ltv_by_channel, labels=[ch.title() for ch in channels],
                         patch_artist=True, widths=0.6)
        colors = sns.color_palette("husl", len(channels))
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        ax.set_title('Lifetime Value Distribution by Channel', fontsize=13, fontweight='bold')
        ax.set_ylabel('LTV ($)')
        ax.set_xlabel('Acquisition Channel')
        ax.grid(True, alpha=0.3, axis='y')
        plt.setp(ax.get_xticklabels(), rotation=15, ha='right')
        fig.tight_layout()
        return fig
 
    def ltv_ratio_heatmap(self):
        profitability = self.analyzer.profitability_matrix()
        fig, ax = plt.subplots(figsize=self.figsize_default)
        sns.heatmap(profitability, cmap='YlOrRd', annot=True, fmt='.0f',
                    cbar_kws={'label': 'LTV / CAC Ratio'}, ax=ax, linewidths=0.5)
        ax.set_title('Profitability Heatmap (LTV/CAC Ratio)', fontsize=13, fontweight='bold')
        ax.set_xlabel('Acquisition Channel')
        ax.set_ylabel('Cohort (Signup Month)')
        plt.setp(ax.get_xticklabels(), rotation=15, ha='right')
        fig.tight_layout()
        return fig
 
    def channel_comparison_dashboard(self):
        fig = plt.figure(figsize=(11, 8))
        gs = GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3)
 
        ax1 = fig.add_subplot(gs[0, 0])
        curves = self.analyzer.all_retention_curves()
        for channel in curves.columns:
            ax1.plot(curves.index, curves[channel] * 100, marker='o', linewidth=1.5, label=channel.title())
        ax1.set_title('Retention Curves', fontsize=11, fontweight='bold')
        ax1.set_xlabel('Months')
        ax1.set_ylabel('Retention (%)')
        ax1.legend(fontsize=7)
        ax1.grid(True, alpha=0.3)
 
        ax2 = fig.add_subplot(gs[0, 1])
        ltv_data = self.analyzer.calculate_ltv_by_cohort_channel()
        channels = sorted(ltv_data['channel'].unique())
        ltv_by_channel = [ltv_data[ltv_data['channel'] == ch]['ltv'].values for ch in channels]
        bp = ax2.boxplot(ltv_by_channel, labels=[ch.title() for ch in channels], patch_artist=True)
        for patch in bp['boxes']:
            patch.set_facecolor('lightblue')
        ax2.set_title('LTV Distribution', fontsize=11, fontweight='bold')
        ax2.set_ylabel('LTV ($)')
        ax2.grid(True, alpha=0.3, axis='y')
        plt.setp(ax2.get_xticklabels(), rotation=15, ha='right', fontsize=8)
 
        ax3 = fig.add_subplot(gs[1, 0])
        payback = self.analyzer.payback_analysis()
        if not payback.empty:
            ax3.barh(payback['channel'], payback['payback_month_mean'], color='coral', alpha=0.7)
        ax3.set_xlabel('Months to CAC Payback')
        ax3.set_title('Avg Payback Period', fontsize=11, fontweight='bold')
        ax3.grid(True, alpha=0.3, axis='x')
 
        ax4 = fig.add_subplot(gs[1, 1])
        channel_summary = self.analyzer.channel_summary()
        ax4.bar(channel_summary['channel'], channel_summary['ltv_ratio_mean'],
                color='lightgreen', alpha=0.7, edgecolor='black')
        ax4.set_title('LTV/CAC Ratio by Channel', fontsize=11, fontweight='bold')
        plt.setp(ax4.get_xticklabels(), rotation=15, ha='right', fontsize=8)
        ax4.grid(True, alpha=0.3, axis='y')
 
        fig.suptitle('Subscription Cohort Analysis Dashboard', fontsize=14, fontweight='bold')
        return fig

 
class ScenarioSimulator:
    def __init__(self, original_df, discount_rate=0.10):
        self.original_df = original_df.copy()
        self.discount_rate = discount_rate
        self.scenarios = {}
 
    def create_scenario(self, name, arpu_multiplier=1.0, retention_improvement=0.0,
                         cac_reduction=0.0, channel_specific=None):
        modified_df = self.original_df.copy()
        modified_df = modified_df.astype({'arpu': 'float64', 'cac': 'float64', 'monthly_revenue': 'float64'})
 
        if arpu_multiplier != 1.0:
            modified_df['arpu'] = modified_df['arpu'] * arpu_multiplier
            modified_df['monthly_revenue'] = modified_df['monthly_revenue'] * arpu_multiplier
 
        if cac_reduction != 0.0:
            modified_df['cac'] = modified_df['cac'] * (1 - cac_reduction)
 
        if retention_improvement != 0.0:
            modified_df['weibull_scale'] = modified_df['weibull_scale'] * (1 + retention_improvement)
            for idx, row in modified_df.iterrows():
                shape = row['weibull_shape']
                scale = row['weibull_scale']
                month = row['months_since_signup']
                new_retention = np.exp(-(month / scale) ** shape)
                modified_df.at[idx, 'retention_rate'] = new_retention
                modified_df.at[idx, 'active_users'] = int(row['cohort_size'] * new_retention)
                modified_df.at[idx, 'monthly_revenue'] = row['arpu'] * modified_df.at[idx, 'active_users']
 
        if channel_specific:
            for channel, params in channel_specific.items():
                mask = modified_df['channel'] == channel
 
                if 'arpu_multiplier' in params:
                    mult = params['arpu_multiplier']
                    modified_df.loc[mask, 'arpu'] = modified_df.loc[mask, 'arpu'] * mult
                    modified_df.loc[mask, 'monthly_revenue'] = (
                        modified_df.loc[mask, 'active_users'] * modified_df.loc[mask, 'arpu']
                    )
 
                if 'cac_reduction' in params:
                    reduction = params['cac_reduction']
                    modified_df.loc[mask, 'cac'] = modified_df.loc[mask, 'cac'] * (1 - reduction)
 
                if 'retention_improvement' in params:
                    improvement = params['retention_improvement']
                    modified_df.loc[mask, 'weibull_scale'] = (
                        modified_df.loc[mask, 'weibull_scale'] * (1 + improvement)
                    )
                    for idx in modified_df[mask].index:
                        row = modified_df.loc[idx]
                        shape = row['weibull_shape']
                        scale = row['weibull_scale']
                        month = row['months_since_signup']
                        new_retention = np.exp(-(month / scale) ** shape)
                        modified_df.at[idx, 'retention_rate'] = new_retention
                        modified_df.at[idx, 'active_users'] = int(row['cohort_size'] * new_retention)
                        modified_df.at[idx, 'monthly_revenue'] = row['arpu'] * modified_df.at[idx, 'active_users']
 
        self.scenarios[name] = {
            'df': modified_df,
            'params': {
                'arpu_multiplier': arpu_multiplier,
                'retention_improvement': retention_improvement,
                'cac_reduction': cac_reduction,
                'channel_specific': channel_specific
            }
        }
        return self
 
    def get_scenario_ltv(self, scenario_name):
        if scenario_name not in self.scenarios:
            raise ValueError(f"Scenario '{scenario_name}' not found")
        df = self.scenarios[scenario_name]['df']
        analyzer = CohortAnalyzer(df, discount_rate=self.discount_rate)
        return analyzer.calculate_ltv_by_cohort_channel()
 
    def compare_scenarios(self, scenarios=None):
        if scenarios is None:
            scenarios = list(self.scenarios.keys())
 
        original_analyzer = CohortAnalyzer(self.original_df, discount_rate=self.discount_rate)
        original_ltv = original_analyzer.calculate_ltv_by_cohort_channel()
 
        comparison = []
        for _, row in original_ltv.groupby('channel').agg({
            'cac': 'first', 'ltv': 'mean', 'ltv_ratio': 'mean', 'payback_month': 'mean'
        }).reset_index().iterrows():
            comparison.append({
                'scenario': 'BASELINE (Original)', 'channel': row['channel'],
                'avg_cac': row['cac'], 'avg_ltv': round(row['ltv'], 2),
                'avg_ltv_ratio': round(row['ltv_ratio'], 2),
                'avg_payback_months': round(row['payback_month'], 1) if not pd.isna(row['payback_month']) else None
            })
 
        for scenario_name in scenarios:
            if scenario_name not in self.scenarios:
                continue
            ltv_df = self.get_scenario_ltv(scenario_name)
            for _, row in ltv_df.groupby('channel').agg({
                'cac': 'first', 'ltv': 'mean', 'ltv_ratio': 'mean', 'payback_month': 'mean'
            }).reset_index().iterrows():
                comparison.append({
                    'scenario': scenario_name, 'channel': row['channel'],
                    'avg_cac': row['cac'], 'avg_ltv': round(row['ltv'], 2),
                    'avg_ltv_ratio': round(row['ltv_ratio'], 2),
                    'avg_payback_months': round(row['payback_month'], 1) if not pd.isna(row['payback_month']) else None
                })
 
        return pd.DataFrame(comparison)
 
    def impact_analysis(self, scenario_name):
        original_analyzer = CohortAnalyzer(self.original_df, discount_rate=self.discount_rate)
        original_ltv = original_analyzer.calculate_ltv_by_cohort_channel()
        scenario_ltv = self.get_scenario_ltv(scenario_name)
 
        impact = []
        for channel in original_ltv['channel'].unique():
            orig = original_ltv[original_ltv['channel'] == channel]
            scen = scenario_ltv[scenario_ltv['channel'] == channel]
            if len(orig) == 0 or len(scen) == 0:
                continue
 
            orig_avg_ltv = orig['ltv'].mean()
            scen_avg_ltv = scen['ltv'].mean()
            orig_avg_ratio = orig['ltv_ratio'].mean()
            scen_avg_ratio = scen['ltv_ratio'].mean()
            orig_payback = orig['payback_month'].mean()
            scen_payback = scen['payback_month'].mean()
 
            ltv_change_pct = ((scen_avg_ltv - orig_avg_ltv) / orig_avg_ltv * 100) if orig_avg_ltv > 0 else 0
            ratio_change_pct = ((scen_avg_ratio - orig_avg_ratio) / orig_avg_ratio * 100) if orig_avg_ratio > 0 else 0
            payback_change_months = (scen_payback - orig_payback) if not np.isnan(orig_payback) else 0
 
            impact.append({
                'channel': channel,
                'baseline_ltv': round(orig_avg_ltv, 2),
                'scenario_ltv': round(scen_avg_ltv, 2),
                'ltv_change_%': round(ltv_change_pct, 1),
                'baseline_ltv_ratio': round(orig_avg_ratio, 2),
                'scenario_ltv_ratio': round(scen_avg_ratio, 2),
                'ratio_change_%': round(ratio_change_pct, 1),
                'baseline_payback_months': round(orig_payback, 1) if not pd.isna(orig_payback) else None,
                'scenario_payback_months': round(scen_payback, 1) if not pd.isna(scen_payback) else None,
                'payback_change_months': round(payback_change_months, 1)
            })
        return pd.DataFrame(impact)

 
def dataframe_to_treeview(parent, df, height=15):
    """Render a pandas DataFrame in a ttk.Treeview inside a frame with scrollbars."""
    frame = ttk.Frame(parent)
    frame.pack(fill='both', expand=True, padx=5, pady=5)
 
    vsb = ttk.Scrollbar(frame, orient='vertical')
    hsb = ttk.Scrollbar(frame, orient='horizontal')
 
    columns = list(df.columns)
    tree = ttk.Treeview(frame, columns=columns, show='headings', height=height,
                         yscrollcommand=vsb.set, xscrollcommand=hsb.set)
    vsb.config(command=tree.yview)
    hsb.config(command=tree.xview)
 
    for col in columns:
        tree.heading(col, text=col)
        tree.column(col, width=110, anchor='center')
 
    for _, row in df.iterrows():
        values = [f"{v:.2f}" if isinstance(v, float) else v for v in row]
        tree.insert('', 'end', values=values)
 
    tree.grid(row=0, column=0, sticky='nsew')
    vsb.grid(row=0, column=1, sticky='ns')
    hsb.grid(row=1, column=0, sticky='ew')
    frame.grid_rowconfigure(0, weight=1)
    frame.grid_columnconfigure(0, weight=1)
    return frame
 
 
def embed_figure(parent, fig):
    """Embed a matplotlib figure into a Tk frame with a navigation toolbar."""
    canvas = FigureCanvasTkAgg(fig, master=parent)
    canvas.draw()
    toolbar = NavigationToolbar2Tk(canvas, parent)
    toolbar.update()
    canvas.get_tk_widget().pack(fill='both', expand=True)
    return canvas
 
 
def clear_frame(frame):
    for widget in frame.winfo_children():
        widget.destroy()

 
class CohortSimulatorApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Subscription Cohort Retention Simulator")
        self.root.geometry("1150x780")
 
        self.generator = None
        self.df = None
        self.analyzer = None
        self.visualizer = None
        self.simulator = None
 
        self.notebook = ttk.Notebook(root)
        self.notebook.pack(fill='both', expand=True)
 
        self.tab_data = ttk.Frame(self.notebook)
        self.tab_analysis = ttk.Frame(self.notebook)
        self.tab_viz = ttk.Frame(self.notebook)
        self.tab_scenario = ttk.Frame(self.notebook)
 
        self.notebook.add(self.tab_data, text='1. Data Generation')
        self.notebook.add(self.tab_analysis, text='2. Cohort Analysis')
        self.notebook.add(self.tab_viz, text='3. Visualizations')
        self.notebook.add(self.tab_scenario, text='4. Scenario Simulator')
 
        self._build_data_tab()
        self._build_analysis_tab()
        self._build_viz_tab()
        self._build_scenario_tab()
 
    def _build_data_tab(self):
        controls = ttk.Frame(self.tab_data, padding=10)
        controls.pack(fill='x')
 
        ttk.Label(controls, text="Number of cohort months:").grid(row=0, column=0, sticky='w', padx=5, pady=5)
        self.n_months_var = tk.IntVar(value=24)
        ttk.Entry(controls, textvariable=self.n_months_var, width=10).grid(row=0, column=1, padx=5, pady=5)
 
        ttk.Label(controls, text="Start date (YYYY-MM-DD):").grid(row=0, column=2, sticky='w', padx=5, pady=5)
        self.start_date_var = tk.StringVar(value='2023-01-01')
        ttk.Entry(controls, textvariable=self.start_date_var, width=14).grid(row=0, column=3, padx=5, pady=5)
 
        ttk.Label(controls, text="Discount rate:").grid(row=0, column=4, sticky='w', padx=5, pady=5)
        self.discount_rate_var = tk.DoubleVar(value=0.10)
        ttk.Entry(controls, textvariable=self.discount_rate_var, width=8).grid(row=0, column=5, padx=5, pady=5)
 
        ttk.Button(controls, text="Generate Data", command=self.generate_data).grid(row=0, column=6, padx=10)
        ttk.Button(controls, text="Load CSV...", command=self.load_csv).grid(row=0, column=7, padx=5)
        ttk.Button(controls, text="Save CSV...", command=self.save_csv).grid(row=0, column=8, padx=5)
 
        self.data_status = ttk.Label(self.tab_data, text="No data generated yet.", padding=10)
        self.data_status.pack(fill='x')
 
        self.data_table_frame = ttk.Frame(self.tab_data)
        self.data_table_frame.pack(fill='both', expand=True)
 
    def generate_data(self):
        try:
            n_months = self.n_months_var.get()
            start_date = self.start_date_var.get()
            discount_rate = self.discount_rate_var.get()
 
            self.generator = SyntheticCohortGenerator(n_months=n_months, start_date=start_date,
                                                        discount_rate=discount_rate)
            self.df = self.generator.generate()
            self._on_data_ready()
        except Exception as e:
            messagebox.showerror("Error generating data", str(e))
 
    def load_csv(self):
        path = filedialog.askopenfilename(filetypes=[("CSV files", "*.csv")])
        if not path:
            return
        try:
            self.df = pd.read_csv(path)
            self._on_data_ready()
        except Exception as e:
            messagebox.showerror("Error loading CSV", str(e))
 
    def save_csv(self):
        if self.df is None:
            messagebox.showwarning("No data", "Generate or load data first.")
            return
        path = filedialog.asksaveasfilename(defaultextension=".csv", filetypes=[("CSV files", "*.csv")])
        if not path:
            return
        self.df.to_csv(path, index=False)
        messagebox.showinfo("Saved", f"Saved to {path}")
 
    def _on_data_ready(self):
        self.data_status.config(text=f"✓ {len(self.df):,} rows | "
                                      f"{self.df['cohort_month'].nunique()} cohorts | "
                                      f"{self.df['channel'].nunique()} channels")
        clear_frame(self.data_table_frame)
        dataframe_to_treeview(self.data_table_frame, self.df.head(100))
 
        discount_rate = self.discount_rate_var.get()
        self.analyzer = CohortAnalyzer(self.df, discount_rate=discount_rate)
        self.visualizer = CohortVisualizer(self.analyzer)
        self.simulator = ScenarioSimulator(self.df, discount_rate=discount_rate)
        self._populate_channel_dropdown()
 
  
    def _build_analysis_tab(self):
        controls = ttk.Frame(self.tab_analysis, padding=10)
        controls.pack(fill='x')
 
        ttk.Button(controls, text="Channel Summary", command=self.show_channel_summary).pack(side='left', padx=5)
        ttk.Button(controls, text="Payback Analysis", command=self.show_payback_analysis).pack(side='left', padx=5)
        ttk.Button(controls, text="LTV by Cohort-Channel", command=self.show_ltv_table).pack(side='left', padx=5)
 
        self.analysis_table_frame = ttk.Frame(self.tab_analysis)
        self.analysis_table_frame.pack(fill='both', expand=True)
 
    def _require_analyzer(self):
        if self.analyzer is None:
            messagebox.showwarning("No data", "Generate or load data in Tab 1 first.")
            return False
        return True
 
    def show_channel_summary(self):
        if not self._require_analyzer():
            return
        clear_frame(self.analysis_table_frame)
        dataframe_to_treeview(self.analysis_table_frame, self.analyzer.channel_summary())
 
    def show_payback_analysis(self):
        if not self._require_analyzer():
            return
        clear_frame(self.analysis_table_frame)
        result = self.analyzer.payback_analysis()
        if result.empty:
            ttk.Label(self.analysis_table_frame, text="No cohorts reached payback within the modeled window.",
                      padding=20).pack()
        else:
            dataframe_to_treeview(self.analysis_table_frame, result)
 
    def show_ltv_table(self):
        if not self._require_analyzer():
            return
        clear_frame(self.analysis_table_frame)
        dataframe_to_treeview(self.analysis_table_frame, self.analyzer.calculate_ltv_by_cohort_channel())
 
 
    def _build_viz_tab(self):
        controls = ttk.Frame(self.tab_viz, padding=10)
        controls.pack(fill='x')
 
        ttk.Label(controls, text="Channel (for heatmap):").pack(side='left', padx=5)
        self.channel_var = tk.StringVar(value='All Channels')
        self.channel_dropdown = ttk.Combobox(controls, textvariable=self.channel_var,
                                              values=['All Channels'], state='readonly', width=15)
        self.channel_dropdown.pack(side='left', padx=5)
 
        ttk.Button(controls, text="Retention Heatmap", command=self.show_retention_heatmap).pack(side='left', padx=5)
        ttk.Button(controls, text="Retention Curves", command=self.show_retention_curves).pack(side='left', padx=5)
        ttk.Button(controls, text="LTV Boxplot", command=self.show_ltv_boxplot).pack(side='left', padx=5)
        ttk.Button(controls, text="Profitability Heatmap", command=self.show_profitability_heatmap).pack(side='left', padx=5)
        ttk.Button(controls, text="Full Dashboard", command=self.show_dashboard).pack(side='left', padx=5)
 
        self.viz_frame = ttk.Frame(self.tab_viz)
        self.viz_frame.pack(fill='both', expand=True)
 
    def _populate_channel_dropdown(self):
        if self.df is not None:
            channels = ['All Channels'] + sorted(self.df['channel'].unique().tolist())
            self.channel_dropdown['values'] = channels
 
    def _require_visualizer(self):
        if self.visualizer is None:
            messagebox.showwarning("No data", "Generate or load data in Tab 1 first.")
            return False
        return True
 
    def _render_figure(self, fig):
        clear_frame(self.viz_frame)
        embed_figure(self.viz_frame, fig)
 
    def show_retention_heatmap(self):
        if not self._require_visualizer():
            return
        channel = None if self.channel_var.get() == 'All Channels' else self.channel_var.get()
        self._render_figure(self.visualizer.retention_heatmap(channel=channel))
 
    def show_retention_curves(self):
        if not self._require_visualizer():
            return
        self._render_figure(self.visualizer.retention_curves_by_channel())
 
    def show_ltv_boxplot(self):
        if not self._require_visualizer():
            return
        self._render_figure(self.visualizer.ltv_by_channel_boxplot())
 
    def show_profitability_heatmap(self):
        if not self._require_visualizer():
            return
        self._render_figure(self.visualizer.ltv_ratio_heatmap())
 
    def show_dashboard(self):
        if not self._require_visualizer():
            return
        self._render_figure(self.visualizer.channel_comparison_dashboard())
 
    
    def _build_scenario_tab(self):
        controls = ttk.LabelFrame(self.tab_scenario, text="Create Scenario", padding=10)
        controls.pack(fill='x', padx=10, pady=10)
 
        ttk.Label(controls, text="Scenario name:").grid(row=0, column=0, sticky='w', padx=5, pady=5)
        self.scenario_name_var = tk.StringVar(value="My Scenario")
        ttk.Entry(controls, textvariable=self.scenario_name_var, width=30).grid(row=0, column=1, padx=5, pady=5, columnspan=3, sticky='w')
 
        ttk.Label(controls, text="ARPU multiplier (1.0 = no change):").grid(row=1, column=0, sticky='w', padx=5, pady=5)
        self.arpu_mult_var = tk.DoubleVar(value=1.0)
        ttk.Entry(controls, textvariable=self.arpu_mult_var, width=10).grid(row=1, column=1, padx=5, pady=5)
 
        ttk.Label(controls, text="Retention improvement (0.1 = +10%):").grid(row=1, column=2, sticky='w', padx=5, pady=5)
        self.retention_var = tk.DoubleVar(value=0.0)
        ttk.Entry(controls, textvariable=self.retention_var, width=10).grid(row=1, column=3, padx=5, pady=5)
 
        ttk.Label(controls, text="CAC reduction (0.15 = -15%):").grid(row=2, column=0, sticky='w', padx=5, pady=5)
        self.cac_reduction_var = tk.DoubleVar(value=0.0)
        ttk.Entry(controls, textvariable=self.cac_reduction_var, width=10).grid(row=2, column=1, padx=5, pady=5)
 
        ttk.Button(controls, text="Create Scenario", command=self.create_scenario).grid(row=2, column=3, padx=5, pady=5)
 
        list_frame = ttk.LabelFrame(self.tab_scenario, text="Scenarios", padding=10)
        list_frame.pack(fill='x', padx=10, pady=5)
 
        self.scenario_listbox = tk.Listbox(list_frame, height=4)
        self.scenario_listbox.pack(side='left', fill='x', expand=True, padx=5)
 
        btns = ttk.Frame(list_frame)
        btns.pack(side='left', padx=10)
        ttk.Button(btns, text="Compare All Scenarios", command=self.show_comparison).pack(fill='x', pady=2)
        ttk.Button(btns, text="Impact Analysis (selected)", command=self.show_impact).pack(fill='x', pady=2)
 
        self.scenario_result_frame = ttk.Frame(self.tab_scenario)
        self.scenario_result_frame.pack(fill='both', expand=True, padx=10, pady=5)
 
    def create_scenario(self):
        if self.simulator is None:
            messagebox.showwarning("No data", "Generate or load data in Tab 1 first.")
            return
        try:
            name = self.scenario_name_var.get().strip()
            if not name:
                messagebox.showwarning("Missing name", "Enter a scenario name.")
                return
            self.simulator.create_scenario(
                name,
                arpu_multiplier=self.arpu_mult_var.get(),
                retention_improvement=self.retention_var.get(),
                cac_reduction=self.cac_reduction_var.get()
            )
            self.scenario_listbox.insert('end', name)
            messagebox.showinfo("Scenario created", f"'{name}' added.")
        except Exception as e:
            messagebox.showerror("Error creating scenario", str(e))
 
    def show_comparison(self):
        if self.simulator is None or not self.simulator.scenarios:
            messagebox.showwarning("No scenarios", "Create at least one scenario first.")
            return
        clear_frame(self.scenario_result_frame)
        dataframe_to_treeview(self.scenario_result_frame, self.simulator.compare_scenarios(), height=12)
 
    def show_impact(self):
        selection = self.scenario_listbox.curselection()
        if not selection:
            messagebox.showwarning("No selection", "Select a scenario from the list first.")
            return
        scenario_name = self.scenario_listbox.get(selection[0])
        clear_frame(self.scenario_result_frame)
        ttk.Label(self.scenario_result_frame, text=f"Impact Analysis: {scenario_name}",
                  font=('TkDefaultFont', 11, 'bold')).pack(anchor='w', pady=(0, 5))
        dataframe_to_treeview(self.scenario_result_frame, self.simulator.impact_analysis(scenario_name), height=10)

In [ ]:
 
if __name__ == "__main__":
    root = tk.Tk()
    app = CohortSimulatorApp(root)
    root.mainloop()